# Native conformal parity, model boundaries and broad-set control


Check every input/output identity, then compare against the unchanged native conformal function on the identical frozen partitions. Its partition chooser is temporarily replaced for this diagnostic and restored; its inference/calibration code is unchanged. Rare classes remain in the full eligible benchmark population. This is prediction-grade recovery, not biological accuracy or an established exchangeability guarantee.

In [1]:
import json, hashlib, ast
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
from starplast import artifacts as A, conformal_records as F, organisms as O, strategies as S, strategy_learning as N, record_scorecards as R, baselines as B
from starplast.splits import read_split
from scripts.freeze_conformal_pilot import frozen_rank_matrix
out=Path('results/conformal_label_pilot_2026_10_08')
m=json.loads((out/'manifest.json').read_text())
sha=lambda path:hashlib.sha256(Path(path).read_bytes()).hexdigest()
for path,digest in m['input_sha256'].items():
    assert sha(path)==digest,path
for path,digest in m['outputs'].items():
    assert sha(out/path)==digest,path
split,exclusions=read_split(out/'split.json')
raw=json.loads((out/'held_out/manifest.json').read_text())
spec=A.spec_from_dict(raw['spec'])
artifact=A.read_artifact(out/'held_out',expected=spec,split=split)
base_spec=A.spec_from_dict(json.loads((out/'base_model/manifest.json').read_text())['spec'])
base=A.read_artifact(out/'base_model',expected=base_spec,split=split)
state=base.payloads['model_state.json']
ids=tuple(a.entity for a in split.assignments)
frame=pd.read_parquet(O.nodes_path(O.TOXOPLASMA)).set_index('gene_id')
matrix=frozen_rank_matrix(frame.loc[list(ids),state['input_columns']],state)
where={entity:i for i,entity in enumerate(ids)}
positions={role:np.array([where[e] for e in split.entities(role)]) for role in ('train','calibration','test')}
np.testing.assert_array_equal(matrix[positions['train']],state['training_vectors'])
visible=pd.Series([None]*len(ids),dtype=object)
for role in ('train','calibration'):
    visible.iloc[positions[role]]=frame.loc[list(split.entities(role)),'compartment'].astype(str).to_numpy()
chooser=N._group_split
N._group_split=lambda *args:(positions['train'],positions['calibration'])
p={'model':'kNN','k':state['k'],'C':.5,'alpha':.1,'thresholds':'per class'}
try:
    native,size,sets,thresholds=N._conformal(S.Context(pd.DataFrame({'gene_id':ids}),graph={},organism=O.TOXOPLASMA),matrix,visible,positions['test'],p)
finally:
    N._group_split=chooser
rows=pd.DataFrame(artifact.payloads['rows.json'],columns=artifact.payloads['row_columns.json'])
expected=native.iloc[positions['test']].astype(object).where(native.iloc[positions['test']].notna(),None).tolist()
assert rows.prediction.astype(object).where(rows.prediction.notna(),None).tolist()==expected
assert rows.native_set_text.tolist()==sets.iloc[positions['test']].tolist()
assert rows.set_size.tolist()==size.iloc[positions['test']].tolist()
score_payload=artifact.payloads['class_scores.json']
scores=pd.DataFrame(score_payload['scores'],columns=score_payload['classes'])
np.testing.assert_array_equal(scores,S.class_scores_of(native).iloc[positions['test']])
calibration=artifact.payloads['calibration.json']
for label,value in thresholds.items():
    assert calibration['thresholds'][label]=={'status':'finite' if np.isfinite(value) else 'unbounded','value':value if np.isfinite(value) else None}
assert hashlib.sha256(A.canonical_object(calibration).encode()).hexdigest()==m['summary']['calibration_identity']
cal_scores=pd.DataFrame(calibration['scores']['scores'],index=calibration['scores']['entities'],columns=calibration['scores']['classes'])
test_scores=scores.copy(); test_scores.index=rows.entity.tolist()
cal_labels=pd.Series(calibration['calibration_labels'],index=calibration['calibration_entities'])
batch=F.conformal_calls(cal_scores,test_scores,cal_labels,split=split,base_model=base,alpha=.1,per_class=True)
assert batch.rows.astype(object).where(batch.rows.notna(),None).to_dict('records')==[{key:row[key] for key in batch.rows.columns} for row in artifact.payloads['rows.json']]
scope_data=dict(artifact.payloads['card.json']['scope']); scope_data['gaps']=tuple(scope_data['gaps'])
scope=R.RecordScope(**scope_data)
normalize=lambda value:json.loads(json.dumps(value,allow_nan=False))
assert normalize(R.aggregate(rows,scope,parameters={'class_scores':scores}))==artifact.payloads['card.json']
assert normalize(R.class_cards(rows,scope,parameters={'class_scores':scores}))==artifact.payloads['class_cards.json']
labels=pd.Series(state['training_labels'],index=state['fit_entities'])
baselines=B.label_baselines(labels,split,seed=split.seed)
for column in ('majority','prevalence_call'):
    simple=rows.drop(columns=['prediction','correct','abstained','prediction_set','native_set_text','set_size']).assign(prediction=baselines[column].to_numpy())
    assert normalize(R.aggregate(simple,scope))==artifact.payloads['baseline_cards.json'][column]
for path in ('starplast/conformal_records.py','scripts/freeze_conformal_pilot.py','starplast/record_scorecards.py'):
    ast.parse(Path(path).read_text(),feature_version=(3,10))
{'native_calls_sets_scores_quantiles':'exact','test_rows':len(rows),'independent_biological_admission':False,'metrics':artifact.payloads['card.json']['metrics']}

{'native_calls_sets_scores_quantiles': 'exact', 'test_rows': 560, 'independent_biological_admission': False, 'metrics': {'accuracy': 0.0, 'coverage': 0.0, 'empty_set_share': 0.0, 'kappa': 0.0, 'macro_auprc': 0.3913263864026719, 'macro_auroc': 0.8167671709952596, 'macro_f1': 0.0, 'macro_precision': 0.0, 'macro_recall': 0.0, 'mcc': 0.0, 'mean_set_size': 24.455357142857142, 'precision_of_calls': None, 'set_coverage': 0.9732142857142857, 'set_efficiency': 0.06178571428571433, 'singleton_share': 0.0, 'weighted_f1': 0.0}}

The all-training-classes control uses only the fitted training class vocabulary. Its broad sets need no calibration truth. Every test gene is retained, and unsupported truth classes would count as set misses. Compare set size/efficiency alongside coverage; a nearly full set is weak inference capacity. The control is not a calibrated deployment promise.

In [2]:
classes=sorted(set(labels),key=str)
assert classes==state['native_classes']
control=rows.copy()
members=classes
control['prediction_set']=[list(members) for _ in range(len(control))]
control['set_size']=len(members)
control['native_set_text']=' | '.join(members)
control['prediction']=pd.Series([members[0] if len(members)==1 else None]*len(control),dtype=object)
control['abstained']=len(members)!=1
control['correct']=pd.array([v==t if isinstance(v,str) else None for v,t in zip(control.prediction,control.truth)],dtype='boolean')
settings=A.canonical_object({'control':'all_training_classes','selection':'complete frozen train vocabulary'})
control_scope=replace(scope,settings=settings)
control_card=R.aggregate(control,control_scope,parameters={'prediction_classes':classes})
verification_code=Path('/media/carruthers/mnt3/claude/repo/starplast/scripts/verify_conformal_pilot.py')
dependencies=base_spec.dependencies+(A.Dependency('model','fixed_training_classes',base.identity),A.Dependency('code','control_verifier',sha(verification_code)))
control_spec=replace(spec,role='scorecard',partition_id='control:all_training_classes:'+split.identity,dependencies=dependencies,settings_json=settings,confidence_kind='none',calibration_scope='',gaps=tuple(sorted(set((*spec.gaps,'All-training-classes control; no calibrated promise')))))
payload={'rows.json':control.astype(object).where(control.notna(),None).to_dict('records'),'row_columns.json':control.columns.tolist(),'card.json':control_card,'class_cards.json':R.class_cards(control,control_scope,parameters={'prediction_classes':classes})}
payload=normalize(payload)
control_identity=A.write_artifact(out/'all_class_control',control_spec,payload,split=split)
assert A.read_artifact(out/'all_class_control',expected=control_spec,split=split).payloads==payload
(out/'code'/verification_code.name).write_bytes(verification_code.read_bytes())
{'control_identity':control_identity,'control_metrics':control_card['metrics'],'class_vocabulary_source':'training only'}

{'control_identity': 'f891a124b6cc40b108ca438733314680cee82c2d1a4c7c760105e88ae9917a45', 'control_metrics': {'accuracy': 0.0, 'coverage': 0.0, 'precision_of_calls': None, 'macro_precision': 0.0, 'macro_recall': 0.0, 'macro_f1': 0.0, 'weighted_f1': 0.0, 'kappa': 0.0, 'mcc': 0.0, 'macro_auroc': None, 'macro_auprc': None, 'set_coverage': 1.0, 'mean_set_size': 26.0, 'singleton_share': 0.0, 'empty_set_share': 0.0, 'set_efficiency': 0.0}, 'class_vocabulary_source': 'training only'}